In [1]:
import pandas as pd

try:
     df = pd.read_csv(r"C:\Users\gabhi\Documents\NIDS-Live-ML\data\processed\live_flows.csv")
     if not df.empty:
         print("df.head()")
except pd.errors.EmptyDataError:
     print("The CSV file is currently empty. Waiting for data....")
     df=pd.DataFrame()

df.head()


In [2]:
print(df.shape)

(26, 16)


In [3]:
X = df.drop(
    columns=[
        "src_ip",
        "dst_ip"
    ], errors="ignore"
)
X.head()


,src_port,dst_port,protocol,packet_count,byte_count,duration,avg_packet_size,packets_per_second,syn_count,ack_count,fin_count,rst_count,syn_ratio,bytes_per_second
0,53,49330,17,6,1596,2.709367,266.000000,2.214539,0,0,0,0,0.000000,589.067423
1,443,56014,6,22,13398,0.890667,609.000000,24.700577,2,21,2,0,0.090909,15042.651635
2,443,58560,6,33,23476,0.942389,711.393939,35.017395,2,32,0,0,0.060606,24911.162647
3,443,63914,6,3,220,0.046923,73.333333,63.934962,0,3,0,0,0.000000,4688.563880
4,60719,3478,17,2,124,5.006657,62.000000,0.399468,0,0,0,0,0.000000,24.767027


SAVING ISOLATION FOREST MODEL

In [10]:
# training the model
from sklearn.ensemble import IsolationForest
import joblib

iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

if X.empty:
    print("No data available to train the model. Waiting for data...")
else:
    iso_model.fit(X)
    # ensure models_dir exists (models_dir is defined in an earlier cell)
    models_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(iso_model, models_dir / "isolation_forest.pkl")
    print(f"Model trained and saved to {models_dir / 'isolation_forest.pkl'}")

Model trained and saved to c:\Users\gabhi\Documents\NIDS-Live-ML\models\isolation_forest.pkl


In [11]:
# checking if model features match prediction features
import joblib
from pathlib import Path

models_path = Path.cwd().parent.parent / "models" / "isolation_forest.pkl"
if not models_path.exists():
    alt_path = models_path.with_name("iso_model.pkl")
    if not models_path.exists():
        if alt_path.exists():
            models_path = alt_path
        else:
            raise FileNotFoundError(f"Model file not found: tried {models_path} and {alt_path}")
model = joblib.load(models_path)

# safe print of feature names
if hasattr(model, "feature_names_in_"):
    print(model.feature_names_in_)
elif hasattr(model, "n_features_in_"):
    print(f"Model trained on {model.n_features_in_} features but feature names not available.")
else:
    print("feature_names_in_ not available (model may be unfitted or trained without feature names).")
if not models_path.exists():
    raise FileNotFoundError(f"Model file not found: {models_path}")
model = joblib.load(models_path)

if hasattr(model, "feature_names_in_"):
    print("Feature names:", model.feature_names_in_)
else:
    print("Model is unfitted or was trained on empty data. No feature information available.")

['src_port' 'dst_port' 'protocol' 'packet_count' 'byte_count' 'duration'
 'avg_packet_size' 'packets_per_second' 'syn_count' 'ack_count'
 'fin_count' 'rst_count' 'syn_ratio' 'bytes_per_second']
Feature names: ['src_port' 'dst_port' 'protocol' 'packet_count' 'byte_count' 'duration'
 'avg_packet_size' 'packets_per_second' 'syn_count' 'ack_count'
 'fin_count' 'rst_count' 'syn_ratio' 'bytes_per_second']


In [13]:
if not models_path.exists():
    if alt_path.exists():
        models_path = alt_path
    else:
        raise FileNotFoundError(f"Model file not found: tried {models_path} and {alt_path}")

model = joblib.load(models_path)

if hasattr(model, "feature_names_in_"):
    print("Feature names:", model.feature_names_in_)
else:
    print("Model is unfitted or was trained on empty data. No feature information available.")
if hasattr(model, "feature_names_in_"):
    print("Feature names:", model.feature_names_in_)
else:
    print("Model is unfitted or was trained on empty data. No feature information available.")

Feature names: ['src_port' 'dst_port' 'protocol' 'packet_count' 'byte_count' 'duration'
 'avg_packet_size' 'packets_per_second' 'syn_count' 'ack_count'
 'fin_count' 'rst_count' 'syn_ratio' 'bytes_per_second']
Feature names: ['src_port' 'dst_port' 'protocol' 'packet_count' 'byte_count' 'duration'
 'avg_packet_size' 'packets_per_second' 'syn_count' 'ack_count'
 'fin_count' 'rst_count' 'syn_ratio' 'bytes_per_second']


In [14]:
if hasattr(iso_model, 'feature_names_in_'):
    print(iso_model.feature_names_in_)
else:
    print("Model has not been fitted yet. No feature names available.")

['src_port' 'dst_port' 'protocol' 'packet_count' 'byte_count' 'duration'
 'avg_packet_size' 'packets_per_second' 'syn_count' 'ack_count'
 'fin_count' 'rst_count' 'syn_ratio' 'bytes_per_second']


In [15]:
# Check if model is fitted and data is available
if hasattr(iso_model, 'feature_names_in_'):
    predictions = iso_model.predict(X)
    df["anomaly"] = predictions
else:
    print("Model not fitted yet. Please ensure data is loaded and model is trained first.")


In [16]:
input_df = X_encoded if "X_encoded" in globals() and not X_encoded.empty else X

if input_df.empty:
    print("No data available to predict anomalies. Ensure X or X_encoded contains rows.")
    predictions = pd.Series(dtype=int)
else:
    predictions = model.predict(input_df)

df["anomaly"] = predictions
df["anomaly"].value_counts()
df["anomaly"] = predictions
df["anomaly"].value_counts()

anomaly
 1    24
-1     2
Name: count, dtype: int64

In [17]:
if "anomaly" not in df.columns:
    if "predictions" in globals():
        df["anomaly"] = predictions
    else:
        print("No anomaly column found. Run the prediction cell first.")

if "anomaly" in df.columns:
    df[df["anomaly"] == -1]